# ADA 4.0: Transformer Sequence Forecasting Brain
Upload `BTC_USDT_1m_dataset.csv` to the root folder of this Colab session before running.


In [ ]:
!pip install pandas-ta scikit-learn tensorflow joblib


## 1. Data Loading and Feature Engineering


In [ ]:
import pandas as pd
import pandas_ta as ta
import numpy as np

# Load Data
df = pd.read_csv('BTC_USDT_1m_dataset.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)

print("Original Data Size:", len(df))

# Feature Engineering
df['returns'] = df['close'].pct_change()
df['volatility'] = df['returns'].rolling(window=20).std()
df['sma_20'] = ta.sma(df['close'], length=20)
df['sma_50'] = ta.sma(df['close'], length=50)

# MACD
macd = ta.macd(df['close'])
df['macd'] = macd['MACD_12_26_9']
df['macd_hist'] = macd['MACDh_12_26_9']

# RSI
df['rsi'] = ta.rsi(df['close'], length=14)

# Drop NaN values generated by rolling windows
df.dropna(inplace=True)
print("Data Size after Feature Engineering:", len(df))


## 2. Sequence Windowing (The Time Machine)
We shape the data into 3D tensors: `(samples, 60_past_candles, features)` to predict `(10_future_returns)`.


In [ ]:
from sklearn.preprocessing import StandardScaler
import joblib

# Select features
features = ['returns', 'volatility', 'sma_20', 'sma_50', 'macd', 'macd_hist', 'rsi']
data = df[features].values

# Target is the 'returns' column (index 0 in our features list)
target_idx = 0 

# Normalize Data (We fit the scaler on the whole dataset for simplicity here, but in production, fit on train only)
scaler = StandardScaler()
data_scaled = scaler.fit_transform(data)
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved as scaler.pkl")

# Create Sequences
LOOKBACK = 60
FORECAST = 10

X, y = [], []
for i in range(len(data_scaled) - LOOKBACK - FORECAST):
    # The past 60 candles (all features)
    X.append(data_scaled[i : i + LOOKBACK])
    # The next 10 candles (just the 'returns' feature)
    y.append(data_scaled[i + LOOKBACK : i + LOOKBACK + FORECAST, target_idx])

X = np.array(X)
y = np.array(y)

print(f"X shape (Samples, TimeSteps, Features): {X.shape}")
print(f"y shape (Samples, ForecastSteps): {y.shape}")

# Chronological Split (80% Train, 20% Test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]


## 3. Build the Transformer Architecture


In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization, MultiHeadAttention, Add, Flatten
from tensorflow.keras.models import Model

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.1):
    # Attention and Normalization
    x = LayerNormalization(epsilon=1e-6)(inputs)
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = Dropout(dropout)(x)
    res = Add()([x, inputs])

    # Feed Forward Part
    x = LayerNormalization(epsilon=1e-6)(res)
    x = Dense(ff_dim, activation="relu")(x)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1])(x)
    return Add()([x, res])

# Build Model
inputs = Input(shape=(LOOKBACK, len(features)))

# 2 Transformer Blocks
x = transformer_encoder(inputs, head_size=32, num_heads=2, ff_dim=32, dropout=0.1)
x = transformer_encoder(x, head_size=32, num_heads=2, ff_dim=32, dropout=0.1)

# Flatten and Output
x = Flatten()(x)
x = Dense(64, activation="relu")(x)
x = Dropout(0.1)(x)
# Output layer predicts 10 future returns
outputs = Dense(FORECAST, activation="linear")(x) 

model = Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="mse", metrics=["mae"])
model.summary()


## 4. Train the Transformer


In [ ]:
print("\nStarting Training...")
# Using a small number of epochs to prevent overfitting and keep Colab fast
history = model.fit(
    X_train, y_train,
    epochs=5, 
    batch_size=256,
    validation_data=(X_test, y_test),
    verbose=1
)


## 5. Export ADA's New Brain


In [ ]:
# Save the trained Keras model
model.save('transformer_brain.keras')
print("Model saved as transformer_brain.keras!")
print("Download both 'transformer_brain.keras' and 'scaler.pkl' and move them to your ADA2 folder.")
